# Notebook 03 — Causal Feature Analysis
**Owner: Person B (CaFE) + Person C (layer evolution, patch/CLS) — Week 2**

Stage 3. Identify causally important features for flamingo vs. spoonbill, run the CaFE sanity check, and compare patch vs. CLS features across layers.

Requires: Person A to have implemented causal.compute_feature_importance().

## 1. Load class-specific activations

In [1]:
"""Step 1a — setup, extract zip, and top up to 200 images per class.

Two-step image sourcing:
  1. Extract data/flamingo_and_spoonbill.zip if the target dirs are empty. The zip is not in the repo.
     Zip contains imagenet_val/flamingo/ and imagenet_val/spoonbill/ entries,
     so extracting to data/ yields data/imagenet_val/flamingo/ and spoonbill/.
  2. utils/load_data.py:download_images() counts existing files, hashes them,
     then streams ImageNet (val → train) to add only unique images up to 200 each.

Requires: pip install -e . (from repo root) so that src.* and utils.* resolve.
"""
import zipfile, json
from pathlib import Path
import torch
from src.config import get_config
import src.config as _cfg_mod

cfg       = get_config()
repo_root = _cfg_mod._DEFAULT_CONFIG.parent.parent

# ── Step 1: extract zip if behavior-class dirs are empty ─────────────────────
data_dir      = repo_root / cfg.data.imagenet_val_path
flamingo_dir  = data_dir / cfg.data.behavior_class_a
spoonbill_dir = data_dir / cfg.data.behavior_class_b
zip_path      = repo_root / "data" / "flamingo_and_spoonbill.zip"

n_fl = sum(1 for _ in flamingo_dir.glob("*.JPEG"))  if flamingo_dir.exists()  else 0
n_sp = sum(1 for _ in spoonbill_dir.glob("*.JPEG")) if spoonbill_dir.exists() else 0

if zip_path.exists() and (n_fl < 10 or n_sp < 10):
    print(f"Extracting {zip_path.name} ...")
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(repo_root / "data")   # imagenet_val/flamingo/ → data/imagenet_val/flamingo/
    n_fl = sum(1 for _ in flamingo_dir.glob("*.JPEG"))
    n_sp = sum(1 for _ in spoonbill_dir.glob("*.JPEG"))
    print(f"Extracted: {n_fl} flamingo, {n_sp} spoonbill")
else:
    print(f"Zip already extracted: {n_fl} flamingo, {n_sp} spoonbill")

# ── Step 2: top up to cfg.data.behavior_n_images with unique streamed images ──
from utils.load_data import download_images

download_images(
    out_dir=str(data_dir),
    n=cfg.data.behavior_n_images,
)

flamingo_paths  = sorted(flamingo_dir.glob("*.JPEG"))
spoonbill_paths = sorted(spoonbill_dir.glob("*.JPEG"))
print(f"\nReady: {len(flamingo_paths)} flamingo, {len(spoonbill_paths)} spoonbill")

Zip already extracted: 174 flamingo, 179 spoonbill
flamingo: 174 images present, 174 unique hashes indexed
spoonbill: 179 images present, 179 unique hashes indexed

Streaming validation split ...


Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

  saved spoonbill_179.JPEG
  saved spoonbill_180.JPEG
  saved spoonbill_181.JPEG
  saved spoonbill_182.JPEG
  saved flamingo_174.JPEG
  saved flamingo_175.JPEG
  saved flamingo_176.JPEG
  saved spoonbill_183.JPEG
  saved spoonbill_184.JPEG
  saved spoonbill_185.JPEG
  saved flamingo_177.JPEG
  saved spoonbill_186.JPEG
  saved flamingo_178.JPEG
  saved flamingo_179.JPEG
  saved flamingo_180.JPEG
  saved flamingo_181.JPEG
  saved spoonbill_187.JPEG
  saved spoonbill_188.JPEG
  saved flamingo_182.JPEG
  saved flamingo_183.JPEG
  saved spoonbill_189.JPEG
  saved spoonbill_190.JPEG
  saved flamingo_184.JPEG
  saved spoonbill_191.JPEG
  saved flamingo_185.JPEG
  saved flamingo_186.JPEG
  saved flamingo_187.JPEG
  saved flamingo_188.JPEG
  saved spoonbill_192.JPEG
  saved spoonbill_193.JPEG
  saved spoonbill_194.JPEG
  saved flamingo_189.JPEG
  saved flamingo_190.JPEG
  saved flamingo_191.JPEG
  saved flamingo_192.JPEG
  saved spoonbill_195.JPEG
  saved flamingo_193.JPEG
  saved flamingo_194.

In [2]:
"""Step 1b — verify classification.

Keeps only images the base model predicts correctly.
logit indices are in cfg.data.flamingo_logit_idx / spoonbill_logit_idx
(confirmed from notebook 01: spoonbill_004.JPEG → [2.18, -0.03] → class 0 wins).
"""
from tqdm.auto import tqdm
from src.model import get_model, preprocess_image

model  = get_model()
device = next(model.parameters()).device

flamingo_idx  = cfg.data.flamingo_logit_idx
spoonbill_idx = cfg.data.spoonbill_logit_idx

def _verify(paths, expected_idx, label):
    correct = []
    for p in tqdm(paths, desc=f"Verifying {label}"):
        pixels = preprocess_image(str(p)).to(device)
        with torch.no_grad():
            logits = model.run_with_hooks(pixels, fwd_hooks=[])
        if logits[0, 0].argmax().item() == expected_idx:
            correct.append(str(p))
    print(f"  {label}: {len(correct)}/{len(paths)} correctly classified")
    return correct

flamingo_ok  = _verify(flamingo_paths,  flamingo_idx,  cfg.data.behavior_class_a)
spoonbill_ok = _verify(spoonbill_paths, spoonbill_idx, cfg.data.behavior_class_b)
print(f"\nVerified dataset: {len(flamingo_ok)} + {len(spoonbill_ok)} = "
      f"{len(flamingo_ok)+len(spoonbill_ok)} images")

C:\Project Explainable\Sparse-Feature-Circuits-in-ViTs\venv\Lib\site-packages\kaleido\_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.

You can however, use the Kaleido API directly which will work with your plotly version. `kaleido.write_fig(...)`, for example. Please see the kaleido documentation.


2026-06-03 16:06:11 INFO:root: Model 'facebook/dino-vitb16' is supported and passes tests.
2026-06-03 16:06:11 DEBUG:urllib3.connectionpool: https://huggingface.co:443 "HEAD /facebook/dino-vitb16/resolve/main/config.json HTTP/1.1" 307 0
2026-06-03 16:06:11 DEBUG:urllib3.connectionpool: https://huggingface.co:443 "HEAD /api/resolve-cache/models/facebook/dino-vitb16/f205d5d8e640a89a2b8ef0369670dfc37cc07fc2/config.json HTTP/1.1" 200 0


ln_pre not set


2026-06-03 16:06:12 DEBUG:urllib3.connectionpool: https://huggingface.co:443 "HEAD /facebook/dino-vitb16/resolve/main/model.safetensors HTTP/1.1" 404 0
2026-06-03 16:06:12 DEBUG:urllib3.connectionpool: https://huggingface.co:443 "HEAD /facebook/dino-vitb16/resolve/main/pytorch_model.bin HTTP/1.1" 302 0
2026-06-03 16:06:12 INFO:root: Filling in 2 missing keys with default initialization
2026-06-03 16:06:12 WARNING:root: Missing key for weight matrix: head.W_H
2026-06-03 16:06:13 INFO:root: Loaded pretrained model facebook/dino-vitb16 into HookedTransformer


Loaded facebook/dino-vitb16 on cuda — 86,389,248 params


Verifying flamingo:   0%|          | 0/200 [00:00<?, ?it/s]

  flamingo: 183/200 correctly classified


Verifying spoonbill:   0%|          | 0/200 [00:00<?, ?it/s]

  spoonbill: 136/200 correctly classified

Verified dataset: 183 + 136 = 319 images


In [3]:
"""Step 1c — extract residual-stream activations for verified images.

Runs one forward pass per image and reads hook_resid_post tensors for every
configured SAE target layer. The behavior-class images may not have been
present when the HDF5 cache was built in notebook 02.
"""
target_layers = list(cfg.sae.target_layers)
hook_keys = {layer: f"blocks.{layer}.hook_resid_post" for layer in target_layers}

def _extract_acts_by_layer(paths, desc):
    acts = {layer: [] for layer in target_layers}
    for p in tqdm(paths, desc=desc):
        pixels = preprocess_image(str(p)).to(device)
        with torch.no_grad():
            _, cache = model.run_with_cache(pixels)
        for layer, hook_key in hook_keys.items():
            acts[layer].append(cache[hook_key].cpu())
    return {layer: torch.cat(layer_acts, dim=0) for layer, layer_acts in acts.items()}

act_flamingo_by_layer = _extract_acts_by_layer(
    flamingo_ok, f"Activations {cfg.data.behavior_class_a}"
)
act_spoonbill_by_layer = _extract_acts_by_layer(
    spoonbill_ok, f"Activations {cfg.data.behavior_class_b}"
)

# Backwards-compatible aliases for the primary-layer CaFE cells below.
act_flamingo = act_flamingo_by_layer[cfg.sae.primary_layer]
act_spoonbill = act_spoonbill_by_layer[cfg.sae.primary_layer]

for layer in target_layers:
    print(
        f"layer {layer}: flamingo {tuple(act_flamingo_by_layer[layer].shape)}, "
        f"spoonbill {tuple(act_spoonbill_by_layer[layer].shape)}"
    )


Activations flamingo:   0%|          | 0/183 [00:00<?, ?it/s]

Activations spoonbill:   0%|          | 0/136 [00:00<?, ?it/s]

layer 4: flamingo (183, 197, 768), spoonbill (136, 197, 768)
layer 6: flamingo (183, 197, 768), spoonbill (136, 197, 768)
layer 9: flamingo (183, 197, 768), spoonbill (136, 197, 768)


## 2. Per-feature importance ranking (Person A)

Coordinate: Person A runs this cell first and saves the scores.

In [ ]:
"""Step 2 — compute feature importance and select top causal features.

Reuses Person A's two-pass ablation loop for every configured SAE target layer.
Runtime: ~15-30 min per layer on A100 for 400 images.
Results are cached to outputs/features/ — re-run will load existing .pt scores.
"""
from src.causal import compute_feature_importance, get_top_causal_features

feature_output_dir = repo_root / "outputs" / "features"
feature_output_dir.mkdir(parents=True, exist_ok=True)

importance_by_layer = {}
top_features_by_layer = {}

for layer in target_layers:
    importance_cache = feature_output_dir / f"importance_layer{layer}.pt"
    ranking_cache = feature_output_dir / f"importance_ranking_layer{layer}.json"

    if importance_cache.exists():
        importance_layer = torch.load(importance_cache, map_location="cpu")
        print(f"Loaded cached importance: {importance_cache}")
    else:
        importance_layer = compute_feature_importance(
            layer,
            act_flamingo_by_layer[layer],
            act_spoonbill_by_layer[layer],
            model,
        )
        torch.save(importance_layer, importance_cache)
        print(f"Saved importance -> {importance_cache}")

    scores = importance_layer.detach().cpu()
    ranked = (scores.abs() > 0).nonzero(as_tuple=True)[0]
    ranked = ranked[torch.argsort(scores[ranked].abs(), descending=True)]
    ranking = [{"feature_idx": int(i), "importance": float(scores[i])} for i in ranked]
    with open(ranking_cache, "w", encoding="utf-8") as f:
        json.dump(ranking, f, indent=2)

    top_features_layer = get_top_causal_features(importance_layer, layer)
    importance_by_layer[layer] = importance_layer
    top_features_by_layer[layer] = top_features_layer

    n_ablated = (importance_layer.abs() > 0).sum().item()
    top5 = [(row["feature_idx"], round(row["importance"], 4)) for row in ranking[:5]]
    print(f"\nLayer {layer}")
    print(f"  Ablated candidates : {n_ablated}  (cfg.causal.grad_top_k = {cfg.causal.grad_top_k})")
    print(f"  Max |importance|   : {importance_layer.abs().max():.4f}")
    print(f"  Top-5 features     : {top5}")
    print(f"  Ranking cache      : {ranking_cache}")
    print(
        f"  Top causal features ({cfg.causal.importance_percentile}th pct of ablated): "
        f"{len(top_features_layer)} features"
    )
    print(f"  Indices: {sorted(top_features_layer)[:20]}{'...' if len(top_features_layer) > 20 else ''}")

importance = importance_by_layer[cfg.sae.primary_layer]
top_features = top_features_by_layer[cfg.sae.primary_layer]


2026-06-03 16:06:39 INFO:root: get_activation_fn received: activation_fn=relu, kwargs={}


Loaded SAE layer 4 — d_in=768, d_sae=49152


Pass 1 — gradient ranking:   0%|          | 0/40 [00:00<?, ?it/s]

Pass 2 — ablation (top 200):   0%|          | 0/200 [00:00<?, ?it/s]

Saved importance -> C:\Project Explainable\Sparse-Feature-Circuits-in-ViTs\outputs\features\importance_layer4.pt

Layer 4
  Ablated candidates : 200  (cfg.causal.grad_top_k = 200)
  Max |importance|   : 0.3951
  Top-5 features     : [(3991, -0.3951), (31694, -0.141), (16973, -0.1173), (13711, -0.1102), (17657, -0.0947)]
  Ranking cache      : C:\Project Explainable\Sparse-Feature-Circuits-in-ViTs\outputs\features\importance_ranking_layer4.json
  Top causal features (80th pct of ablated): 40 features
  Indices: [1657, 2052, 3463, 3644, 3991, 7649, 8202, 8396, 9452, 10896, 11386, 13711, 14764, 14885, 16546, 16973, 17346, 17657, 18391, 21687]...


...

Pass 2 — ablation (top 200):   0%|          | 0/200 [00:00<?, ?it/s]

2026-06-03 16:18:07 INFO:root: get_activation_fn received: activation_fn=relu, kwargs={}


Saved importance -> C:\Project Explainable\Sparse-Feature-Circuits-in-ViTs\outputs\features\importance_layer6.pt

Layer 6
  Ablated candidates : 200  (cfg.causal.grad_top_k = 200)
  Max |importance|   : 0.3881
  Top-5 features     : [(9293, -0.3881), (2218, 0.379), (3574, -0.2266), (43491, -0.1189), (38534, -0.1086)]
  Ranking cache      : C:\Project Explainable\Sparse-Feature-Circuits-in-ViTs\outputs\features\importance_ranking_layer6.json
  Top causal features (80th pct of ablated): 40 features
  Indices: [617, 724, 2218, 2622, 3574, 4462, 8747, 9293, 11844, 14173, 14438, 14612, 15247, 20575, 21131, 21702, 23847, 23901, 25583, 25897]...
Loaded SAE layer 9 — d_in=768, d_sae=49152


Pass 1 — gradient ranking:   0%|          | 0/40 [00:00<?, ?it/s]

Pass 2 — ablation (top 200):   0%|          | 0/200 [00:00<?, ?it/s]

## 3. Ablation ranking plot

In [ ]:
"""Step 3 — ablation ranking plots.

Horizontal bar charts of the top-20 causally important features per target layer.
"""
from src.visualise import plot_ablation_ranking

figures_dir = repo_root / cfg.outputs.report_figures_path
figures_dir.mkdir(parents=True, exist_ok=True)

for layer in target_layers:
    labels_path = feature_output_dir / f"clip_labels_layer{layer}_full.json"
    if labels_path.exists():
        with open(labels_path, encoding="utf-8") as f:
            feature_labels = {int(k): v for k, v in json.load(f).items()}
    else:
        feature_labels = {}
        print(f"Warning: CLIP labels not found for layer {layer} — run notebook 02 first.")

    importance_dict = {
        fi: importance_by_layer[layer][fi].item()
        for fi in top_features_by_layer[layer]
    }
    save_path = figures_dir / f"ablation_ranking_layer{layer}.png"
    fig = plot_ablation_ranking(
        importance_dict,
        feature_labels=feature_labels,
        top_n=20,
        layer=layer,
        save_path=str(save_path),
    )
    display(fig)
    print(f"Saved layer {layer} -> {save_path}")


## 4. CaFE sanity check (Person B)

Run on top 10 causally important features. Report agreement rate.

Reference: Han et al. (2025) arXiv:2509.00749

In [ ]:
from src.causal import cafe_sanity_check
from src.visualise import plot_cafe_comparison

cafe_output_dir = feature_output_dir / "cafe"
cafe_output_dir.mkdir(parents=True, exist_ok=True)

cafe_feature_indices = sorted(
    top_features,
    key=lambda fi: importance[fi].abs().item(),
    reverse=True,
)[:10]

cafe_results_by_feature = {}
for feat_idx in cafe_feature_indices:
    result = cafe_sanity_check(
        cfg.sae.primary_layer,
        feat_idx,
        act_flamingo,
        flamingo_ok,
        model,
        top_k=10,
    )
    cafe_results_by_feature[feat_idx] = result
    print(f"Feature {feat_idx}: CaFE agreement = {result['agreement_rate']:.2f}")

    result_path = cafe_output_dir / f"cafe_layer{cfg.sae.primary_layer}_feat{feat_idx}.json"
    with open(result_path, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2)

    fig = plot_cafe_comparison(
        result,
        feat_idx,
        save_path=str(figures_dir / f"cafe_layer{cfg.sae.primary_layer}_feat{feat_idx}.png"),
    )
    display(fig)
    print(f"Saved -> {result_path}")


## 5. Patch vs. CLS feature comparison (Person C)

In [ ]:
# TODO (Person C):
# Re-run top patch retrieval with token_type="cls"
# Compare causally important features between patch and CLS.
# Key question: do CLS features encode more global/semantic concepts?
# Document in report/notes/patch_vs_cls.md

## 6. Layer-wise evolution (Person C)

In [ ]:
from src.visualise import plot_layer_evolution
# TODO (Person C):
# Run feature annotation for layers 6 and 9 (repeat notebook 02 steps)
# category_counts = {6: {...}, 9: {...}, 11: {...}}
# fig = plot_layer_evolution(
#     category_counts,
#     save_path=cfg.outputs.report_figures_path + "/layer_evolution.png"
# )

## Checkpoint

**Person A (cells 1a–3)**
- [ ] Zip extracted — flamingo/spoonbill dirs populated
- [ ] Classification verified — counts logged (expect ~150–180 correct per class)
- [ ] Activations extracted for layers 4, 6, and 9
- [ ] `importance_layer{4,6,9}.pt` and `importance_ranking_layer{4,6,9}.json` saved
- [ ] `top_features_by_layer` non-empty for each layer
- [ ] `ablation_ranking_layer{4,6,9}.png` saved

**Person B (cell 4)**
- [ ] cafe_sanity_check implemented in src/causal.py
- [ ] Agreement rate reported for top-10 primary-layer features
- [ ] CaFE figures saved

**Person C (cells 5–6)**
- [ ] Patch vs. CLS comparison documented in report/notes/patch_vs_cls.md
- [ ] Layer evolution figure saved
